# Deliverable 6 — Factorized collision oracle + FT scenario estimates

Tests flat sparsity, implements the useful block/Kronecker factorization of the 90-dimensional collision map, and exposes transparent FT resource scenarios.

This notebook is an executable evidence artifact. Its default configuration is
deliberately small enough for a clean local rerun; scale-up parameters are
listed separately and are not represented as measured results.

In [1]:
from pathlib import Path
import sys

repo_root = Path.cwd().parent if Path.cwd().name == "deliverables" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
output_dir = repo_root / "results" / "deliverables"
output_dir.mkdir(parents=True, exist_ok=True)

## Scope

This notebook first tests whether an ordinary sparse encoding is justified.
It then validates the useful structure: store the local \(R\) and \(Q\)
factors and evaluate the lower block as \(R F_2 R^T\), without materializing
\(R\otimes R\). This is not yet a compiled PREPARE/SELECT block encoding, so
all FT numbers are labeled estimates and their assumptions are stored.

In [2]:
import json
import math
import pandas as pd
from quantum_aero.deliverables import sparse_collision_oracle

oracle = sparse_collision_oracle(omega=1.2)
assert oracle["oracle_matvec_max_error"] < 1e-11
assert oracle["factorized_matvec_max_error"] < 1e-11
oracle

{'dimension': 90,
 'nnz': 7138,
 'density': 0.8812345679012346,
 'max_row_sparsity': 81,
 'median_row_sparsity': 81.0,
 'unique_coefficient_magnitudes': 35,
 'oracle_matvec_max_error': 8.881784197001252e-16,
 'factorized_matvec_max_error': 1.4432899320127035e-15,
 'r_nnz': 81,
 'q_nnz': 496,
 'factorized_stored_coefficients': 577,
 'flat_to_factorized_storage_ratio': 12.370883882149046,
 'column_index_bits': 7,
 'row_count_bits': 7}

In [3]:
# Explicit proxy: PREPARE/SELECT stores the nonzero coefficients of R and Q;
# the R⊗R action reuses R twice. This is deliberately conservative and easy
# to replace when a compiled circuit exists.
coefficient_bits = 16
factorized_terms = oracle["factorized_stored_coefficients"]
address_bits = math.ceil(math.log2(factorized_terms))
logical_qubits = 8 + address_bits + oracle["column_index_bits"] + coefficient_bits + 4
toffoli_per_query = 4 * factorized_terms + 4 * coefficient_bits * 9
t_per_query = 4 * toffoli_per_query

scenarios = [
    {"name": "optimistic", "physical_error": 1e-4, "code_distance": 15, "cycle_us": 0.2,
     "factory_qubits": 12000, "t_states_per_cycle": 4.0},
    {"name": "base", "physical_error": 1e-3, "code_distance": 25, "cycle_us": 1.0,
     "factory_qubits": 30000, "t_states_per_cycle": 1.0},
    {"name": "pessimistic", "physical_error": 3e-3, "code_distance": 35, "cycle_us": 2.0,
     "factory_qubits": 60000, "t_states_per_cycle": 0.25},
]
rows = []
for s in scenarios:
    data_qubits = 2 * logical_qubits * s["code_distance"]**2
    t_cycles = t_per_query / s["t_states_per_cycle"]
    rows.append({**s, "logical_qubits_proxy": logical_qubits,
                 "T_count_per_block_query_proxy": t_per_query,
                 "physical_qubits_proxy": data_qubits + s["factory_qubits"],
                 "block_query_time_seconds_proxy": t_cycles * s["cycle_us"] * 1e-6})
df = pd.DataFrame(rows)
df

,name,physical_error,code_distance,cycle_us,factory_qubits,t_states_per_cycle,logical_qubits_proxy,T_count_per_block_query_proxy,physical_qubits_proxy,block_query_time_seconds_proxy
0,optimistic,0.0001,15,0.2,12000,4.00,45,11536,32250,0.000577
1,base,0.0010,25,1.0,30000,1.00,45,11536,86250,0.011536
2,pessimistic,0.0030,35,2.0,60000,0.25,45,11536,170250,0.092288


In [4]:
payload = {
    "oracle": oracle,
    "assumptions": {
        "coefficient_bits": coefficient_bits,
        "toffoli_per_query_formula": "4*factorized_stored_coefficients + 4*coefficient_bits*9",
        "T_per_Toffoli": 4,
        "surface_code_data_qubits": "2*logical_qubits*distance^2",
        "warning": "proxy only; excludes state preparation, amplification, streaming, measurement, and a compiled PREPARE/SELECT circuit",
    },
    "scenarios": rows,
}
(output_dir / "06_structured_collision_ft_estimates.json").write_text(json.dumps(payload, indent=2))
print("PASS: factorized R/Q implementation exactly reproduces the dense collision matvec.")
print("Flat matrix density:", oracle["density"], "(flat sparsity is not useful)")
print("Flat/factorized storage ratio:", oracle["flat_to_factorized_storage_ratio"])

PASS: factorized R/Q implementation exactly reproduces the dense collision matvec.
Flat matrix density: 0.8812345679012346 (flat sparsity is not useful)
Flat/factorized storage ratio: 12.370883882149046
